# ECG Classification - PyTorch Transformer (v5-pytorch)✅ PyTorch Implementation✅ Fixed Data Leakage✅ Stronger Regularization✅ Direct ONNX Export

## STEP 1: Imports

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderfrom sklearn.preprocessing import StandardScaler, LabelEncoderfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import (    confusion_matrix, ConfusionMatrixDisplay, classification_report,    roc_auc_score, roc_curve, precision_score, recall_score, f1_score,    accuracy_score, log_loss, mean_absolute_error, mean_squared_error, r2_score,    silhouette_score, davies_bouldin_score)from sklearn.utils.class_weight import compute_class_weightimport warningswarnings.filterwarnings('ignore')print(f'PyTorch version: {torch.__version__}')device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Device: {device}')# Set random seeds for reproducibilityRANDOM_STATE = 42torch.manual_seed(RANDOM_STATE)np.random.seed(RANDOM_STATE)if torch.cuda.is_available():    torch.cuda.manual_seed(RANDOM_STATE)

## STEP 2: Load Data

In [ ]:
# Load data - supports both Kaggle and local environments# Same data loading as original TensorFlow version for consistencytry:    # Kaggle environment    df1 = pd.read_csv('/kaggle/input/ecg-dataset/ecg.csv', header=None)    df2 = pd.read_csv('/kaggle/input/ecg2-dataset/ecg3.csv', header=None)    df2 = df2.rename(columns={0: 'orig_0'})    df2.insert(0, 0, df2['orig_0'])    df2.columns = range(df2.shape[1])    df = pd.concat([df1, df2], ignore_index=True)except:    try:        # Local environment - try repository root        df1 = pd.read_csv('../../ecg.csv', header=None)        df2 = pd.read_csv('../../ecg3.csv', header=None)        df2 = df2.rename(columns={0: 'orig_0'})        df2.insert(0, 0, df2['orig_0'])        df2.columns = range(df2.shape[1])        df = pd.concat([df1, df2], ignore_index=True)    except:        df = pd.read_csv('../../dataset_aritmia_NEW.csv')# Add meaningful column namesn_features = df.shape[1] - 1column_names = [f'f{i}' for i in range(n_features)] + ['label']df.columns = column_namesprint(f'Dataset shape: {df.shape}')print(f'Total samples: {df.shape[0]}')print(f'Features per sample: {n_features}')print(f'\nLabel distribution:')print(df['label'].value_counts())

## STEP 3: Preprocessing - **FIXED DATA LEAKAGE**

In [ ]:
X = df.drop('label', axis=1).valuesy = df['label'].values# Split FIRSTX_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp)# Normalize - fit only on trainingscaler = StandardScaler()X_train_norm = scaler.fit_transform(X_train)X_val_norm = scaler.transform(X_val)X_test_norm = scaler.transform(X_test)# Reshape for TransformerX_train_r = X_train_norm.reshape(-1, 188, 1)X_val_r = X_val_norm.reshape(-1, 188, 1)X_test_r = X_test_norm.reshape(-1, 188, 1)print(f'Train: {X_train_r.shape}, Val: {X_val_r.shape}, Test: {X_test_r.shape}')

## STEP 4: Dataset

In [ ]:
class ECGDataset(Dataset):    def __init__(self, X, y):        self.X = torch.FloatTensor(X)        self.y = torch.LongTensor(y)    def __len__(self): return len(self.X)    def __getitem__(self, idx): return self.X[idx], self.y[idx]train_ds = ECGDataset(X_train_r, y_train)val_ds = ECGDataset(X_val_r, y_val)test_ds = ECGDataset(X_test_r, y_test)BATCH_SIZE = 32train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)print(f'Loaders: {len(train_loader)} batches')

## STEP 5: Transformer Model**Regularization:** dropout=0.5

In [ ]:
class PositionalEncoding(nn.Module):    """Learnable positional encoding"""    def __init__(self, d_model, max_len=188):        super().__init__()        self.encoding = nn.Parameter(torch.randn(max_len, d_model))        def forward(self, x):        seq_len = x.size(1)        return x + self.encoding[:seq_len, :]class ECG_Transformer(nn.Module):    """    Transformer Model for ECG Classification (PyTorch version)        Architecture:    - Input projection to higher dimension    - Learnable positional encoding    - 3 Transformer encoder layers with multi-head attention    - Global average pooling    - Dense layers for classification    """    def __init__(self, d_model=64, nhead=4, num_layers=3, num_classes=2, dropout=0.5):        super().__init__()                # Input projection        self.input_proj = nn.Linear(1, d_model)                # Positional encoding        self.pos_encoding = PositionalEncoding(d_model)                # Transformer encoder        encoder_layer = nn.TransformerEncoderLayer(            d_model=d_model,            nhead=nhead,            dim_feedforward=256,            dropout=dropout,            batch_first=True        )        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)                # Global pooling and classification head        self.global_pool = nn.AdaptiveAvgPool1d(1)                self.fc = nn.Sequential(            nn.Linear(d_model, 128),            nn.ReLU(),            nn.Dropout(dropout),            nn.Linear(128, 64),            nn.ReLU(),            nn.Dropout(dropout),            nn.Linear(64, num_classes)        )        def forward(self, x):        # Input projection        x = self.input_proj(x)                # Add positional encoding        x = self.pos_encoding(x)                # Transformer encoding        x = self.transformer(x)                # Global pooling        x = x.permute(0, 2, 1)  # (batch, d_model, seq_len)        x = self.global_pool(x).squeeze(-1)  # (batch, d_model)                # Classification        x = self.fc(x)                return x# Create modelmodel = ECG_Transformer(d_model=64, nhead=4, num_layers=3, num_classes=2, dropout=0.5).to(device)print('Transformer Model Architecture:')print(model)print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters())}')print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')

## STEP 6: Training Config

In [ ]:
# Compute class weights for imbalanced dataclass_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)class_weights = torch.FloatTensor(class_weights).to(device)print(f'Class weights: {class_weights}')criterion = nn.CrossEntropyLoss(weight=class_weights)optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5, min_lr=1e-6, verbose=True)print('Training configuration ready')

## STEP 7: Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):    model.train()    loss_sum, correct, total = 0, 0, 0    for X, y in loader:        X, y = X.to(device), y.to(device)        optimizer.zero_grad()        out = model(X)        loss = criterion(out, y)        loss.backward()        optimizer.step()        loss_sum += loss.item()        _, pred = out.max(1)        total += y.size(0)        correct += pred.eq(y).sum().item()    return loss_sum/len(loader), correct/totaldef val_epoch(model, loader, criterion, device):    model.eval()    loss_sum, correct, total = 0, 0, 0    with torch.no_grad():        for X, y in loader:            X, y = X.to(device), y.to(device)            out = model(X)            loss = criterion(out, y)            loss_sum += loss.item()            _, pred = out.max(1)            total += y.size(0)            correct += pred.eq(y).sum().item()    return loss_sum/len(loader), correct/total

## STEP 8: Training Loop

In [ ]:
NUM_EPOCHS, PATIENCE = 100, 15best_loss, counter = float('inf'), 0train_losses, val_losses, train_accs, val_accs = [], [], [], []print('Starting Training...')print('=' * 70)for e in range(NUM_EPOCHS):    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)    val_loss, val_acc = val_epoch(model, val_loader, criterion, device)        # Store metrics    train_losses.append(tr_loss)    val_losses.append(val_loss)    train_accs.append(tr_acc)    val_accs.append(val_acc)        scheduler.step(val_loss)    current_lr = optimizer.param_groups[0]['lr']        if (e+1) % 5 == 0 or e < 5:        print(f'Epoch [{e+1}/{NUM_EPOCHS}] Train: {tr_acc:.4f} Val: {val_acc:.4f} LR: {current_lr:.6f}')        if val_loss < best_loss:        best_loss = val_loss        counter = 0        torch.save({'model': model.state_dict(), 'epoch': e, 'val_acc': val_acc, 'val_loss': val_loss}, 'ecg_transformer_pytorch_best.pth')    else:        counter += 1        if counter >= PATIENCE:            print(f'\nEarly stopping at epoch {e+1}')            breakprint('\nTraining Complete!')print('=' * 70)cp = torch.load('ecg_transformer_pytorch_best.pth')model.load_state_dict(cp['model'])print(f'Best model from epoch {cp["epoch"]+1}')print(f'Best validation accuracy: {cp["val_acc"]:.4f}')

## STEP 8b: Visualize Training History

In [ ]:
# =============================================================================# Visualize Training History# =============================================================================fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Loss plotaxes[0].plot(train_losses, label='Training Loss', linewidth=2)axes[0].plot(val_losses, label='Validation Loss', linewidth=2)axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Loss')axes[0].set_title('Training and Validation Loss - Transformer (v5)')axes[0].legend()axes[0].grid(True, alpha=0.3)# Accuracy plotaxes[1].plot(train_accs, label='Training Accuracy', linewidth=2)axes[1].plot(val_accs, label='Validation Accuracy', linewidth=2)axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Accuracy')axes[1].set_title('Training and Validation Accuracy - Transformer (v5)')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

## STEP 9: Evaluation

In [ ]:
# =============================================================================# STEP 9: Comprehensive Model Evaluation on TEST SET# =============================================================================print('=' * 70)print('COMPREHENSIVE MODEL EVALUATION - Transformer (v5-pytorch)')print('=' * 70)# Get predictions and probabilitiesmodel.eval()all_preds, all_probs, all_labels = [], [], []with torch.no_grad():    for inputs, labels in test_loader:        outputs = model(inputs.to(device))        probs = F.softmax(outputs, dim=1)        _, predicted = outputs.max(1)        all_preds.extend(predicted.cpu().numpy())        all_probs.extend(probs.cpu().numpy())        all_labels.extend(labels.numpy())all_preds = np.array(all_preds)all_probs = np.array(all_probs)all_labels = np.array(all_labels)# =============================================================================# CATEGORY 1: Basic Classification Metrics# =============================================================================print('\n' + '=' * 70)print('CATEGORY 1: BASIC CLASSIFICATION METRICS')print('=' * 70)accuracy = accuracy_score(all_labels, all_preds)precision = precision_score(all_labels, all_preds, average='weighted')recall = recall_score(all_labels, all_preds, average='weighted')f1 = f1_score(all_labels, all_preds, average='weighted')logloss = log_loss(all_labels, all_probs)print(f'\n1. ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)')print(f'2. PRECISION: {precision:.4f}')print(f'3. RECALL: {recall:.4f}')print(f'4. F1 SCORE: {f1:.4f}')print(f'5. LOG LOSS: {logloss:.4f}')# =============================================================================# CATEGORY 2: ROC Curve and AUC# =============================================================================print('\n' + '=' * 70)print('CATEGORY 2: ROC CURVE AND AUC METRICS')print('=' * 70)auc_score = roc_auc_score(all_labels, all_probs[:, 1])fpr, tpr, thresholds = roc_curve(all_labels, all_probs[:, 1])tnr = 1 - fprfnr = 1 - tproptimal_idx = np.argmax(tpr - fpr)optimal_threshold = thresholds[optimal_idx]print(f'\n6. AUC-ROC: {auc_score:.4f}')print(f'   TPR at optimal threshold: {tpr[optimal_idx]:.4f}')print(f'   TNR at optimal threshold: {tnr[optimal_idx]:.4f}')print(f'   FPR at optimal threshold: {fpr[optimal_idx]:.4f}')print(f'   FNR at optimal threshold: {fnr[optimal_idx]:.4f}')# Plot ROC curve and Confusion Matrixfig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {auc_score:.4f})')axes[0].plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier')axes[0].scatter(fpr[optimal_idx], tpr[optimal_idx], c='green', s=100, label=f'Optimal Point', zorder=5)axes[0].set_xlabel('False Positive Rate')axes[0].set_ylabel('True Positive Rate')axes[0].set_title('ROC Curve - Transformer (v5-pytorch)')axes[0].legend()axes[0].grid(True, alpha=0.3)cm = confusion_matrix(all_labels, all_preds)disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal (0)', 'Abnormal (1)'])disp.plot(ax=axes[1], cmap='Blues', values_format='d')axes[1].set_title('Confusion Matrix')plt.tight_layout()plt.show()# =============================================================================# CATEGORY 3: Error Metrics# =============================================================================print('\n' + '=' * 70)print('CATEGORY 3: ERROR METRICS')print('=' * 70)y_true_proba = np.ones(len(all_labels))y_pred_proba_true = all_probs[np.arange(len(all_labels)), all_labels]mae = mean_absolute_error(y_true_proba, y_pred_proba_true)mse = mean_squared_error(y_true_proba, y_pred_proba_true)rmse = np.sqrt(mse)y_pred_safe = np.clip(y_pred_proba_true, 1e-10, 1)rmsle = np.sqrt(np.mean((np.log1p(y_pred_safe) - np.log1p(y_true_proba))**2))r2 = r2_score(y_true_proba, y_pred_proba_true)print(f'\n8a. MAE: {mae:.6f}')print(f'8b. MSE: {mse:.6f}')print(f'8c. RMSE: {rmse:.6f}')print(f'8d. RMSLE: {rmsle:.6f}')print(f'8e. R²: {r2:.6f}')# =============================================================================# CATEGORY 4: Clustering Metrics# =============================================================================print('\n' + '=' * 70)print('CATEGORY 4: CLUSTERING METRICS')print('=' * 70)try:    X_test_flat = X_test_reshaped.reshape(len(X_test_reshaped), -1)    silhouette = silhouette_score(X_test_flat, all_preds)    dbi = davies_bouldin_score(X_test_flat, all_preds)    print(f'\n9a. Silhouette Score: {silhouette:.4f}')    print(f'9b. Davies-Bouldin Index: {dbi:.4f}')except Exception as e:    print(f'Could not compute clustering metrics: {e}')# =============================================================================# Classification Report# =============================================================================print('\n' + '=' * 70)print('DETAILED CLASSIFICATION REPORT')print('=' * 70)print(classification_report(all_labels, all_preds, target_names=['Normal (0)', 'Abnormal (1)']))# =============================================================================# SUMMARY# =============================================================================print('\n' + '=' * 70)print('SUMMARY - TRANSFORMER MODEL (v5-pytorch)')print('=' * 70)print(f'Accuracy:    {accuracy:.4f}')print(f'Precision:   {precision:.4f}')print(f'Recall:      {recall:.4f}')print(f'F1 Score:    {f1:.4f}')print(f'AUC-ROC:     {auc_score:.4f}')

## STEP 10: Save & Export

In [ ]:
torch.save({'model': model.state_dict(), 'test_acc': test_acc}, 'ecg_transformer_v5_pytorch_final.pth')import joblibjoblib.dump(scaler, 'scaler_v5_pytorch.pkl')print('Models saved')try:    model.eval()    dummy = torch.randn(1, 188, 1).to(device)    torch.onnx.export(model, dummy, 'ecg_transformer_v5_pytorch_final.onnx',                     export_params=True, opset_version=13,                     input_names=['input'], output_names=['output'],                     dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})    print('✓ ONNX exported: ecg_transformer_v5_pytorch_final.onnx')    import onnxruntime as ort    ort.InferenceSession('ecg_transformer_v5_pytorch_final.onnx')    print('✓ ONNX verified')except Exception as e:    print(f'ONNX export failed: {e}')